# 검수 확인서 테스트

In [1]:
import sys
import os
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain_teddynote import logging
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from pathlib import Path
from IPython.display import Markdown, display

from dotenv import load_dotenv

load_dotenv(override=True)

# 추적을 위한 프로젝트 이름 설정
logging.langsmith("Samsung-Asset-AI-Portal")

# prompt 모듈에서 필요한 항목 import
from prompt import (
    _SYSTEM_PROMPT,
    _SYSTEM_PROMPT_ENG,
    get_prompt_pdf_text_to_markdown,
    get_prompt_text_to_markdown,
    get_prompt_pdf_text_to_markdown_validate,
    get_prompt_text_to_markdown_validate,
    get_prompt_confirmed_expected_clarification,
    get_prompt_text_to_markdown_validate_report,
    get_prompt_text_to_markdown_error_fix,
    get_prompt_confirmed_expected_clarification_validate_report,
    get_prompt_confirmed_expected_clarification_error_fix
)

# document_parser 모듈에서 필요한 함수들 import
from document_parser import (
    extract_pdf_with_docling,
    extract_pdf_with_pdfplumber,
    parser_excel,
    get_file_path,
    get_last_ai_message
)

from utils import format_messages

_original_text = None

LangSmith 추적을 시작합니다.
[프로젝트명]
Samsung-Asset-AI-Portal


# LLM 모델 생성

In [2]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

def create_llm_model():

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    # llm = init_chat_model(
    #     "openai:gpt-4o",
    #     temperature=0.0,
    #     top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    # )
    return llm

# excel to markdown

In [3]:
# 싱글톤 
def text_to_markdown_with_llm(document_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown(document_text)
    print("########## text to markdown prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_with_plan(document_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown(document_text)
    print("########## text to markdown prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

def convert_excel_to_markdown(file_path: str):
    # 엑셀 파일 경로 확인
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # 엑셀 파일에서 텍스트 추출
    _original_text = parser_excel(file_path)

    # 텍스트를 markdown으로 변환
    # response_markdown = text_to_markdown_with_plan(document_text_excel)
    response_markdown = text_to_markdown_with_llm(_original_text)
    # markdown_text = get_last_ai_message(response_markdown)

    return response_markdown


# document to markdown

In [4]:
def document_to_markdown(file_path: str):
    path_obj = Path(file_path)
    file_name = path_obj.name  # 파일명 (확장자 포함)
    file_ext = path_obj.suffix  # 확장자 (점 포함, 예: .pdf)
    markdown_text = None

    if file_ext == ".xlsx" or file_ext == ".xls":
        markdown_text = convert_excel_to_markdown(file_path)
    elif file_ext == ".pdf":
        print("pdf - 공사중...")
    else:
        print("지원하지 않는 파일 형식입니다.")

    return markdown_text


# markdown 정리 문서 검수 보고서 작성

In [5]:
# 싱글톤 
def text_to_markdown_validate_report_with_llm(markdown_text: str, original_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_validate_report(markdown_text, original_text)
    print("########## 검수보고서 작성 prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_validate_report_with_plan(markdown_text: str, original_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown_validate_report(markdown_text, original_text)
    print("########## 검수보고서 작성 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# markdown 정리 문서 오류 수정

In [6]:
# 싱글톤 
def text_to_markdown_error_fix_with_llm(markdown_text: str, validate_report: str, original_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_error_fix(markdown_text, original_text, validate_report)
    print("########## 오류 수정 prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_error_fix_with_plan(markdown_text: str, validate_report: str, original_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown_error_fix(markdown_text, original_text, validate_report)
    print("########## 오류 수정 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 정리

In [7]:
# markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
# 싱글톤
def classify_confirmed_expected_from_instruction_with_llm(instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 구분 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_from_instruction_with_plan(instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)
    print("########## 확정분/청구분 구분 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 분류 결과 검수

In [8]:
# 싱글톤
def classify_confirmed_expected_validate_report_with_llm(clarification_markdown: str, instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification_validate_report(clarification_markdown, instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 검수 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_validate_report_with_plan(clarification_markdown: str, instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification_validate_report(clarification_markdown, instruction_markdown)
    print("########## 확정분/청구분 검수 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 오류 수정

In [9]:
# 싱글톤
def classify_confirmed_expected_error_fix_with_llm(validate_report: str, clarification_markdown: str, instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification_error_fix(validate_report, clarification_markdown, instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 오류 수정 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_error_fix_with_plan(validate_report: str, clarification_markdown: str, instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification_error_fix(validate_report, clarification_markdown, instruction_markdown)
    print("########## 확정분/청구분 오류 수정 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# TEST

In [10]:
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
# _password = '345678'



### markdown 변환

In [11]:
# result_markdown = document_to_markdown(_document_file_path)

_original_text = parser_excel(_document_file_path)
result_markdown = text_to_markdown_with_llm(_original_text)

excel_path: /Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx
using pandas
########## text to markdown prompt ##########

  아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 파일에서 추출한 data입니다.

  아래의 추출 data를 LLM 모델이 잘 이해할 수 있도록 정리 지침에 따라 정리하세요.

  정리 지침에 따라 정리한 내용을 출력 형식에 따라 작성하세요.



  ### 변액일임펀드 설정/해지 지시서 파일 내용 ###

  [=== 시트: 당일 ===
 |  |  |  |  |  |  |  |  |  |  |  |  | 
■ Closing Date  :  | 2025-08-25 00:00:00 |  |  |  |  |  |  |  |  |  |  |  | 
■ 결제일          :  | 2025-08-26 00:00:00 |  |  |  |  |  |  |  |  |  |  |  | 발신자 : 라이나생명보험주식회사
 |  |  |  |  |  |  |  |  |  |  |  |  | 
운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액
삼성자산 | 6104 | 채권형 | KB은행 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 | 0 | 0
삼성자산 | 6108 | 유럽주식형 | KB은행 | 3489683178 | 17594 | 59667661 | 1108.34 | 19500 | 66132056 | 3430033111 | 20250826 | 1187413601 | 1316057991
삼성자산 | 6114 | 인덱스주식형 | KB은행 | 1792484830 | 4275

In [12]:
print(f"file_name : {_document_file_path}")
display(Markdown(result_markdown.content))

file_name : /Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx


1. 메타 데이터 정리 테이블

| 일련번호 | 항목 | 값 | 비고 |
|----------|------|----|------|
| 1 | 문서 제목 | 변액일임펀드 설정/해지 지시서 | 시트명: 당일 |
| 2 | Closing Date | 2025-08-25 00:00:00 | 결제 전 기준일 |
| 3 | 결제일 | 2025-08-26 00:00:00 | 실제 자금 결제일 |
| 4 | 발신자 | 라이나생명보험주식회사 | |
| 5 | 집합투자업자(Collective Investment Business Entity) | LINA Insurance Co. | |
| 6 | 신탁업자(Trust Business Entity) | KB Bank | |
| 7 | 일반사무관리회사(General Administration Company) | HANA Investors Service | |
| 8 | 투자일임운용사(Discretionary Investment Asset Mgt. Co.) | 3 AMC | |
| 9 | 투자일임운용사 상세 | Korea Inv, Shinhan, Samsung | |
| 10 | KB Bank 설정금액 | 1959157539 | 수익자(라이나)가 지급할 금액 |
| 11 | KB Bank 해지금액(LINA 수취금액) | 446673328 | 수익자(라이나)가 수취할 금액 |
| 12 | KB Bank NET 금액 | 1512484211 | 설정금액 - 해지금액 (KB Bank 기준 정산액) |

2. 펀드 거래 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 6104 | 채권형 | KB은행 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 | 0 | 0 |
| 2 | 삼성자산 | 6108 | 유럽주식형 | KB은행 | 3489683178 | 17594 | 59667661 | 1108.34 | 19500 | 66132056 | 3430033111 | 20250826 | 1187413601 | 1316057991 |
| 3 | 삼성자산 | 6114 | 인덱스주식형 | KB은행 | 1792484830 | 427550 | 20361 | 2586.83 | 1106000 | 52671 | 1792892019 | 20250826 | 0 | 0 |
| 4 | 삼성자산 | 6115 | 삼성그룹주형 | KB은행 | 436582138 | 101709 | 3119 | 1437.93 | 146250 | 4485 | 436680728 | 20250826 | 0 | 0 |
| 5 | 삼성자산,한투 | 6101 | 혼합성장형 | KB은행 | 8141715654 | 5687732 | 1240755 | 4556.44 | 25915800 | 5653417 | 8146162631 | 20250826 | 0 | 0 |
| 6 | 신한BNP | 6105 | 해외혼합형 | KB은행 | 282648356 | 154566 | 2883 | 1712.86 | 264750 | 4939 | 282800039 | 20250826 | 0 | 0 |
| 7 | 신한BNP | 6106 | 애그리비즈니스주식형 | KB은행 | 1373108730 | 1104931 | 15977 | 1054.41 | 1165050 | 16846 | 1374197684 | 20250826 | 0 | 0 |
| 8 | 신한BNP | 6107 | 아시아50주식형 | KB은행 | 1406890167 | 465143 | 375119 | 2106.88 | 980000 | 790329 | 1406980191 | 20250826 | 0 | 0 |
| 9 | 신한BNP | 6109 | 기후변화주식형 | KB은행 | 52079311 | 0 | 0 | 1737.47 | 0 | 0 | 52079311 | 20250826 | 0 | 0 |
| 10 | 신한BNP | 610J | 밸류고배당주식형 | KB은행 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 | 0 | 0 |
| 11 | 신한BNP | 610L | 미국주식형 | KB은행 | 2032544521 | 507226 | 432988 | 1757.6 | 891500 | 761019 | 2032618759 | 20250826 | 0 | 0 |
| 12 | 신한BNP | 6110 | 차이나주식형 | KB은행 | 15125229377 | 2662577910 | 426437881 | 721.37 | 1920703825 | 307619493 | 17361369406 | 20250826 | 0 | 0 |
| 13 | 신한BNP | 6111 | 글로벌이머징형 | KB은행 | 1471152392 | 805855 | 13021 | 1529.71 | 1232725 | 19919 | 1471945226 | 20250826 | 0 | 0 |
| 14 | 신한BNP | 6112 | 이머징커머더티주식형 | KB은행 | 548143851 | 0 | 841 | 1508.2 | 0 | 1268 | 548143010 | 20250826 | 0 | 0 |
| 15 | 신한BNP | 6113 | 주식형 | KB은행 | 2386104162 | 573999 | 8471011 | 2895.04 | 1661750 | 24523917 | 2378207150 | 20250826 | 0 | 0 |
| 16 | 신한BNP | 6118 | 동남아시아형 | KB은행 | 16270639 | 0 | 0 | 1113.8 | 0 | 0 | 16270639 | 20250826 | 0 | 0 |
| 17 | 한국투신 | 6102 | 혼합안정형 | KB은행 | 1643221367 | 1446095 | 126965 | 2646.19 | 3826639 | 335969 | 1644540497 | 20250826 | 0 | 0 |
| 18 | 한국투신 | 6103 | 브이캡그로스형 | KB은행 | 174005603 | 0 | 6340 | 1520.38 | 0 | 9639 | 173999263 | 20250826 | 0 | 0 |
| 19 | 한국투신 | 610K | 해외혼합&시니어론형 | KB은행 | 31248561 | 0 | 0 | 1185.38 | 0 | 0 | 31248561 | 20250826 | 0 | 0 |
| 20 | 한국투신 | 6116 | 스마트&세이프코스피원자재형 | KB은행 | 357834644 | 0 | 16211 | 1116.98 | 0 | 18107 | 357818433 | 20250826 | 0 | 0 |
| 21 | 한국투신 | 6117 | 스마트&세이프코스피항셍형 | KB은행 | 119098188 | 0 | 0 | 1052.83 | 0 | 0 | 119098188 | 20250826 | 0 | 0 |

3. 합계 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 삼성자산_합계 |  |  | 11098676039 | 1265800 | 82791031 | 0 | 2515500 | 106151097 | 11017150808 |  | 1187413601 | 1316057991 |
| 2 | 삼성자산,한투 | 삼성자산_한투_합계 |  |  | 8141715654 | 5687732 | 1240755 | 0 | 25915800 | 5653417 | 8146162631 |  | 0 | 0 |
| 3 | 신한BNP | 신한BNP_합계 |  |  | 28150335341 | 2666189630 | 436212580 | 0 | 1926899600 | 334505099 | 30380312391 |  | 0 | 0 |
| 4 | 한국투신 | 한국투신_합계 |  |  | 2325408363 | 1446095 | 149516 | 0 | 3826639 | 363715 | 2326704942 |  | 0 | 0 |
| 5 | 합계(운용사) |  |  |  | 49716135397 | 2674589257 | 520393882 | 0 | 1959157539 | 446673328 | 51870330772 |  | 1187413601 | 1316057991 |

In [13]:
# format_messages(result_markdown["messages"])

In [14]:
# print(f"file_name : {_document_file_path}")
# last_ai_message = get_last_ai_message(result_markdown)
# print(last_ai_message)
# display(Markdown(last_ai_message.content))

### 검수

In [15]:
# print("####### original_text #######")
# print(_original_text)

In [16]:
validate_report = text_to_markdown_validate_report_with_llm(result_markdown.content, _original_text)
# print(validate_report)

########## 검수보고서 작성 prompt ##########

    아래의 변액일임펀드 설정/해지 지시서 분석 정리 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시서 원본 파일 문서를 분석하여 정리한 문서입니다.

    원본 파일 문서와 비교하여 분석 정리 문서가가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 아래의 출력 형식에 따라 검수 결과 보고서를 작성하여 출력하세요.



    # 검수 지침

    [
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확인하세요.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확인하세요.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 모든 테이블의 첫번째 컬럼에 일련 번호가 추가되어 있는지 확인하세요.

  - 펀드 거래 정보와 합계 정보가 분리되어 있는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 있는지 확인하세요.

  - 기타 오류 사항이 있는지 확인하세요.
]



    # 변액일임펀드 설정/해지 지시서 분석 정리 문서

    [1. 메타 데이터 정리 테이블

| 일련번호 | 항목 | 값 | 비고 |
|----------|------|----|------|
| 1 | 문서 제목 | 변액일임펀드 설정/해지 지시서 | 시트명: 당일

In [17]:
# validate_report = get_last_ai_message(validate_report)
# display(Markdown(validate_report.content))

In [18]:
display(Markdown(validate_report.content))

1. 검수 결과 보고서

| 일련번호 | 검수 항목 | 검수 내용 | 근거 및 수정 사항 |
|----------|-----------|-----------|------------------|
| 1 | 한자 변환 | 없음 | 원문에 한자 없음. 모든 텍스트는 한글/영문으로 구성되어 있어 변환 필요 없음. |
| 2 | 메타데이터 누락 | 있음 | 원본 문서에 “- Korea Inv, Shinhan, Samsung”이 줄바꿈 후 별도 행으로 존재하나, 분석 정리 문서의 항목 9 “투자일임운용사 상세”에 “Korea Inv, Shinhan, Samsung”으로 정리됨. 그러나 원본에는 “Korea Inv,  Shinhan, Samsung”(공백 2개)로 기재되어 있어, 띄어쓰기 오류가 존재함. → 수정: “Korea Inv,  Shinhan, Samsung” (공백 2개 유지) |
| 3 | 펀드명 공백/띄어쓰기 오류 | 있음 | 원본에서 “브이캡그로스형  ”(끝에 공백 2개)로 기재됨. 분석 정리 문서는 “브이캡그로스형”으로 정리하여 공백 제거. → 원본의 공백은 의미 있는 데이터일 수 있으므로 유지해야 함. 수정: “브이캡그로스형  ” (공백 2개 유지) |
| 4 | 펀드코드와 펀드명 혼동 | 있음 | 합계 정보 테이블의 1행: 운용사명=삼성자산, 펀드코드=삼성자산_합계, 펀드명=공백 → 정상. 그러나 2행: 운용사명=삼성자산,한투, 펀드코드=삼성자산_한투_합계, 펀드명=공백 → 정상. 그러나 3행: 신한BNP_합계, 4행: 한국투신_합계도 정상. 그러나 **합계(운용사)** 행의 펀드코드와 펀드명이 모두 공백으로 기재되어 있으나, 원본에서도 공백이므로 정상. |
| 5 | 테이블 병합/중복 | 있음 | 분석 정리 문서의 펀드 거래 정보 테이블 5행: 운용사명=“삼성자산,한투”, 펀드코드=“6101” → 원본도 동일. 그러나 합계 정보 테이블 2행: 운용사명=“삼성자산,한투”, 펀드코드=“삼성자산_한투_합계” → 이는 별도의 합계 행으로 정상. 그러나 **합계 정보 테이블 1행**의 “전일좌수”, “설정신청좌수”, “해지좌수” 값이 **삼성자산 단일 펀드 합계**가 아니라, **삼성자산 + 삼성자산,한투의 혼합합계**로 보임. → 원본에서 “삼성자산_합계” 행은 삼성자산 단일 운용사 펀드만 합산한 값(11098676039), “삼성자산,한투_합계”는 별도로 존재. 그러나 합계 정보 테이블 1행의 값(11098676039)은 삼성자산 단일 펀드 합계와 동일하므로, **합계 정보 테이블 1행은 삼성자산 단일 펀드 합계**로 해석 가능. 그러나 **합계 정보 테이블 2행**의 전일좌수(8141715654)는 “삼성자산,한투”의 6101 펀드의 전일좌수와 동일. → 이는 **합계 정보 테이블 2행이 합계가 아니라, 단일 펀드(6101)의 데이터를 복사한 것**으로 오류. → **합계 정보 테이블 2행은 합계가 아니며, 단일 펀드 데이터가 잘못 병합됨**. 수정: 합계 정보 테이블 2행은 삭제 또는 수정 필요. |
| 6 | 합계 정보 테이블 오류 | 있음 | 합계 정보 테이블 2행: 운용사명=삼성자산,한투, 펀드코드=삼성자산_한투_합계, 전일좌수=8141715654 → 이는 6101 혼합성장형의 전일좌수와 동일. 즉, **합계가 아니라 단일 펀드 데이터가 합계 행으로 오류 입력됨**. 원본에서도 이 행은 “삼성자산,한투 | 삼성자산_한투_합계 |  |  | 8141715654 | 5687732 | 1240755 | 0 | 25915800 | 5653417 | 8146162631 |  | 0 | 0”로 기재되어 있으나, 이는 **6101 펀드의 데이터**이며, **합계 행이 아님**. → **합계 정보 테이블 2행은 합계가 아니라 단일 펀드 행으로 분류되어야 함**. → **합계 정보 테이블에 잘못된 행이 포함됨**. |
| 7 | 결제일 형식 오류 | 있음 | 분석 정리 문서의 펀드 거래 정보 테이블에서 결제일은 “20250826”으로 숫자형으로 기재. 원본도 동일. 그러나 메타데이터의 결제일은 “2025-08-26 00:00:00”으로 날짜 형식. → **일관성 결여**. 분석 정리 문서는 펀드 거래 정보의 결제일을 “20250826”으로 유지하되, 메타데이터는 “2025-08-26 00:00:00”으로 유지 → 이는 정상. (시스템 입력 시 형식 차이 허용) |
| 8 | 미처리좌수/금액 합계 오류 | 있음 | 합계 정보 테이블 5행의 미처리좌수=1187413601, 미처리금액=1316057991 → 이 값은 **삼성자산 6108 유럽주식형**의 미처리좌수/금액과 동일. 원본에서도 이 값은 유럽주식형에만 존재. → **합계 정보 테이블의 미처리좌수/금액은 합계가 아니라, 단일 펀드의 값이 복사되어 합계 행에 기재됨**. → **합계 정보 테이블의 미처리좌수/금액은 모든 펀드의 합계가 아니라, 단일 펀드의 값이 잘못 병합됨**. → **오류**. |
| 9 | 펀드코드 유효성 | 있음 | “610J”, “610L” 등 알파벳 포함 펀드코드는 원본 그대로 유지. 약어로 사용되므로 정상. |
| 10 | 테이블 일련번호 | 정상 | 모든 테이블에 일련번호 1부터 순차 부여됨. |
| 11 | 펀드 거래 정보와 합계 정보 분리 | 부분 오류 | 펀드 거래 정보 테이블에는 단일 펀드 행만 포함되어야 하나, **합계 정보 테이블 2행이 단일 펀드 행(6101)으로 오류 포함됨**. → **합계 정보 테이블에 단일 펀드 행이 혼입됨**. → **분리 불완전**. |
| 12 | Markdown 오류 | 없음 | 테이블 형식이 올바르게 작성됨. 병합 셀, 불완전한 경계 없음. |
| 13 | 기타 오류 | 있음 | 원본에서 “KB Bank 설정금액” 등은 별도 행으로 기재되었으나, 분석 정리 문서의 메타데이터에 정상 반영됨. 그러나 “- Korea Inv,  Shinhan, Samsung”의 **공백 2개**가 분석 정리 문서에서 **공백 1개**로 축약됨. → 원본의 띄어쓰기 오류는 유지해야 함. 수정 필요. |

2. 검수 결과 점수  
VALIDATE_RESULT:FAIL

### 검수 결과 오류 수정

In [19]:
fixed_markdown = text_to_markdown_error_fix_with_llm(result_markdown.content, validate_report.content, _original_text)
# print(fixed_markdown)

########## 오류 수정 prompt ##########

    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서에 대한 분석 정리 문서를 검수하여, 검수 결과를 정리한 검수 결과 보고서입니다.

    검수 결과 보고서에 작성된 오류 사항을 확인하여 오류가 있으면 변액일임펀드 설정/해지 지시서 분석 정리 문서를 수정하세요.    

    오류 수정 시, 정보가 부족하면 변액일임펀드 설정/해지 지시서 분석 정리 문서 작성 지침을 참고하세요.

    오류 수정 시, 정보가 부족하면 변액일임펀드 설정/해지 지시서 원본 파일 문서를 참고하세요.

    아래의 출력 형식에 따라 출력하세요.

    # 검수 결과 보고서

    [1. 검수 결과 보고서

| 일련번호 | 검수 항목 | 검수 내용 | 근거 및 수정 사항 |
|----------|-----------|-----------|------------------|
| 1 | 한자 변환 | 없음 | 원문에 한자 없음. 모든 텍스트는 한글/영문으로 구성되어 있어 변환 필요 없음. |
| 2 | 메타데이터 누락 | 있음 | 원본 문서에 “- Korea Inv, Shinhan, Samsung”이 줄바꿈 후 별도 행으로 존재하나, 분석 정리 문서의 항목 9 “투자일임운용사 상세”에 “Korea Inv, Shinhan, Samsung”으로 정리됨. 그러나 원본에는 “Korea Inv,  Shinhan, Samsung”(공백 2개)로 기재되어 있어, 띄어쓰기 오류가 존재함. → 수정: “Korea Inv,  Shinhan, Samsung” (공백 2개 유지) |
| 3 | 펀드명 공백/띄어쓰기 오류 | 있음 | 원본에서 “브이캡그로스형  ”(끝에 공백 2개)로 기재됨. 분석 정리 문서는 “브이캡그로스형”으로 정리하여 공백 제거. → 원본의 공백은 의미 있는 데이터일 수 있으므로 유지해야 함. 수정: “브이캡그로스형  ” (공백 2개 유지) |
| 4 | 펀드코드와 펀드명 혼동 | 있음 | 합계 정보 테이블의

In [20]:
display(Markdown(fixed_markdown.content))

[1. 메타 데이터 정리 테이블

| 일련번호 | 항목 | 값 | 비고 |
|----------|------|----|------|
| 1 | 문서 제목 | 변액일임펀드 설정/해지 지시서 | 시트명: 당일 |
| 2 | Closing Date | 2025-08-25 00:00:00 | 결제 전 기준일 |
| 3 | 결제일 | 2025-08-26 00:00:00 | 실제 자금 결제일 |
| 4 | 발신자 | 라이나생명보험주식회사 | |
| 5 | 집합투자업자(Collective Investment Business Entity) | LINA Insurance Co. | |
| 6 | 신탁업자(Trust Business Entity) | KB Bank | |
| 7 | 일반사무관리회사(General Administration Company) | HANA Investors Service | |
| 8 | 투자일임운용사(Discretionary Investment Asset Mgt. Co.) | 3 AMC | |
| 9 | 투자일임운용사 상세 | Korea Inv,  Shinhan, Samsung | |
| 10 | KB Bank 설정금액 | 1959157539 | 수익자(라이나)가 지급할 금액 |
| 11 | KB Bank 해지금액(LINA 수취금액) | 446673328 | 수익자(라이나)가 수취할 금액 |
| 12 | KB Bank NET 금액 | 1512484211 | 설정금액 - 해지금액 (KB Bank 기준 정산액) |

2. 펀드 거래 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 6104 | 채권형 | KB은행 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 | 0 | 0 |
| 2 | 삼성자산 | 6108 | 유럽주식형 | KB은행 | 3489683178 | 17594 | 59667661 | 1108.34 | 19500 | 66132056 | 3430033111 | 20250826 | 1187413601 | 1316057991 |
| 3 | 삼성자산 | 6114 | 인덱스주식형 | KB은행 | 1792484830 | 427550 | 20361 | 2586.83 | 1106000 | 52671 | 1792892019 | 20250826 | 0 | 0 |
| 4 | 삼성자산 | 6115 | 삼성그룹주형 | KB은행 | 436582138 | 101709 | 3119 | 1437.93 | 146250 | 4485 | 436680728 | 20250826 | 0 | 0 |
| 5 | 삼성자산,한투 | 6101 | 혼합성장형 | KB은행 | 8141715654 | 5687732 | 1240755 | 4556.44 | 25915800 | 5653417 | 8146162631 | 20250826 | 0 | 0 |
| 6 | 신한BNP | 6105 | 해외혼합형 | KB은행 | 282648356 | 154566 | 2883 | 1712.86 | 264750 | 4939 | 282800039 | 20250826 | 0 | 0 |
| 7 | 신한BNP | 6106 | 애그리비즈니스주식형 | KB은행 | 1373108730 | 1104931 | 15977 | 1054.41 | 1165050 | 16846 | 1374197684 | 20250826 | 0 | 0 |
| 8 | 신한BNP | 6107 | 아시아50주식형 | KB은행 | 1406890167 | 465143 | 375119 | 2106.88 | 980000 | 790329 | 1406980191 | 20250826 | 0 | 0 |
| 9 | 신한BNP | 6109 | 기후변화주식형 | KB은행 | 52079311 | 0 | 0 | 1737.47 | 0 | 0 | 52079311 | 20250826 | 0 | 0 |
| 10 | 신한BNP | 610J | 밸류고배당주식형 | KB은행 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 | 0 | 0 |
| 11 | 신한BNP | 610L | 미국주식형 | KB은행 | 2032544521 | 507226 | 432988 | 1757.6 | 891500 | 761019 | 2032618759 | 20250826 | 0 | 0 |
| 12 | 신한BNP | 6110 | 차이나주식형 | KB은행 | 15125229377 | 2662577910 | 426437881 | 721.37 | 1920703825 | 307619493 | 17361369406 | 20250826 | 0 | 0 |
| 13 | 신한BNP | 6111 | 글로벌이머징형 | KB은행 | 1471152392 | 805855 | 13021 | 1529.71 | 1232725 | 19919 | 1471945226 | 20250826 | 0 | 0 |
| 14 | 신한BNP | 6112 | 이머징커머더티주식형 | KB은행 | 548143851 | 0 | 841 | 1508.2 | 0 | 1268 | 548143010 | 20250826 | 0 | 0 |
| 15 | 신한BNP | 6113 | 주식형 | KB은행 | 2386104162 | 573999 | 8471011 | 2895.04 | 1661750 | 24523917 | 2378207150 | 20250826 | 0 | 0 |
| 16 | 신한BNP | 6118 | 동남아시아형 | KB은행 | 16270639 | 0 | 0 | 1113.8 | 0 | 0 | 16270639 | 20250826 | 0 | 0 |
| 17 | 한국투신 | 6102 | 혼합안정형 | KB은행 | 1643221367 | 1446095 | 126965 | 2646.19 | 3826639 | 335969 | 1644540497 | 20250826 | 0 | 0 |
| 18 | 한국투신 | 6103 | 브이캡그로스형  | KB은행 | 174005603 | 0 | 6340 | 1520.38 | 0 | 9639 | 173999263 | 20250826 | 0 | 0 |
| 19 | 한국투신 | 610K | 해외혼합&시니어론형 | KB은행 | 31248561 | 0 | 0 | 1185.38 | 0 | 0 | 31248561 | 20250826 | 0 | 0 |
| 20 | 한국투신 | 6116 | 스마트&세이프코스피원자재형 | KB은행 | 357834644 | 0 | 16211 | 1116.98 | 0 | 18107 | 357818433 | 20250826 | 0 | 0 |
| 21 | 한국투신 | 6117 | 스마트&세이프코스피항셍형 | KB은행 | 119098188 | 0 | 0 | 1052.83 | 0 | 0 | 119098188 | 20250826 | 0 | 0 |

3. 합계 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 삼성자산_합계 |  |  | 11098676039 | 1265800 | 82791031 | 0 | 2515500 | 106151097 | 11017150808 |  | 0 | 0 |
| 2 | 신한BNP | 신한BNP_합계 |  |  | 28150335341 | 2666189630 | 436212580 | 0 | 1926899600 | 334505099 | 30380312391 |  | 0 | 0 |
| 3 | 한국투신 | 한국투신_합계 |  |  | 2325408363 | 1446095 | 149516 | 0 | 3826639 | 363715 | 2326704942 |  | 0 | 0 |
| 4 | 합계(운용사) |  |  |  | 49716135397 | 2674589257 | 520393882 | 0 | 1959157539 | 446673328 | 51870330772 |  | 0 | 0 |

### 재 검수

In [21]:
validate_report = text_to_markdown_validate_report_with_llm(fixed_markdown.content, _original_text)
# print(validate_report)

########## 검수보고서 작성 prompt ##########

    아래의 변액일임펀드 설정/해지 지시서 분석 정리 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시서 원본 파일 문서를 분석하여 정리한 문서입니다.

    원본 파일 문서와 비교하여 분석 정리 문서가가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 아래의 출력 형식에 따라 검수 결과 보고서를 작성하여 출력하세요.



    # 검수 지침

    [
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확인하세요.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확인하세요.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 모든 테이블의 첫번째 컬럼에 일련 번호가 추가되어 있는지 확인하세요.

  - 펀드 거래 정보와 합계 정보가 분리되어 있는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 있는지 확인하세요.

  - 기타 오류 사항이 있는지 확인하세요.
]



    # 변액일임펀드 설정/해지 지시서 분석 정리 문서

    [[1. 메타 데이터 정리 테이블

| 일련번호 | 항목 | 값 | 비고 |
|----------|------|----|------|
| 1 | 문서 제목 | 변액일임펀드 설정/해지 지시서 | 시트명: 당

In [22]:
display(Markdown(validate_report.content))

1. 검수 결과 보고서

| 일련번호 | 검수 항목 | 검수 내용 | 근거 및 수정 사항 |
|----------|-----------|-----------|------------------|
| 1 | 한자 변환 | 없음 | 원문에 한자가 전혀 발견되지 않음. |
| 2 | 메타데이터 누락 | 있음 | 원문에 “Prepared by”, “Checked by”, “Approved by”가 존재하나 분석 정리 문서에 누락됨. 이 항목들은 문서의 공식적 검토 프로세스를 나타내므로 반드시 포함되어야 함. |
| 3 | 펀드명 공백/띄어쓰기 오류 | 있음 | “브이캡그로스형  ” (2행, 18번)에 끝에 공백 1개 존재. 원문에도 동일한 공백이 있으나, 의미상 오류로 판단되어 제거해야 함. → “브이캡그로스형”으로 수정. |
| 4 | 운용사명 병합 오류 | 있음 | 원문에서 “삼성자산,한투”는 두 운용사의 공동 운용을 의미하나, 분석 정리 문서에서는 “삼성자산,한투”로 정상 기재됨. 그러나 “삼성자산_한투_합계”라는 합계 행이 원문에 존재함에도 분석 정리 문서에서는 누락됨. → 합계 행을 추가해야 함. |
| 5 | 합계 행 누락 | 있음 | 원문에 “삼성자산_한투_합계” 합계 행이 존재함 (행 번호: 삼성자산,한투 다음). 그러나 분석 정리 문서의 펀드 거래 정보 테이블에 이 행이 전혀 포함되지 않음. → 반드시 추가해야 함. |
| 6 | 합계 행 펀드코드 오류 | 있음 | 분석 정리 문서의 합계 정보 테이블에서 “삼성자산_합계” 행의 펀드코드는 “삼성자산_합계”로 기재되어 있으나, 원문에서는 “삼성자산_합계”가 펀드코드 컬럼에 위치함. 이는 정상. 그러나 “삼성자산_한투_합계”는 원문에 존재하나 분석 정리 문서에 누락됨. → 합계 정보 테이블에 추가 필요. |
| 7 | 합계 정보 테이블의 결제일 누락 | 있음 | 합계 정보 테이블의 결제일 컬럼이 모두 공백으로 기재됨. 원문에서도 합계 행의 결제일은 공백이므로 정상이나, 테이블 구조상 일관성 유지 차원에서 “20250826”을 입력해야 함. 그러나 원문이 공백이므로 “공백” 유지가 원칙. → 원문 그대로 유지. |
| 8 | 미처리좌수/미처리금액 합계 오류 | 있음 | 원문의 “합계(운용사)” 행에서 미처리좌수와 미처리금액은 “1187413601”과 “1316057991”로 기재됨. 그러나 분석 정리 문서의 합계 정보 테이블에서는 이 값이 “0”으로 잘못 기재됨. → 이는 심각한 오류. 원문의 “합계(운용사)” 행의 미처리좌수/금액은 “삼성자산”의 유럽주식형 펀드에서만 발생한 미처리이므로, 합계 행에 그 값을 그대로 반영해야 함. → 수정 필요. |
| 9 | 펀드코드와 펀드명의 혼동 | 있음 | “삼성자산_합계”와 “신한BNP_합계” 등은 펀드코드 컬럼에 기재된 합계 식별자이며, 펀드명은 공백. 분석 정리 문서는 이를 정확히 구분하여 기재함. → 정상. |
| 10 | 펀드코드 유효성 | 있음 | “610J”, “610L” 등 알파벳 포함 펀드코드는 원문 그대로 기재되어 있으며, 약어로 사용됨. → 정상. |
| 11 | 테이블 첫 컬럼 일련번호 | 있음 | 모든 테이블에 일련번호가 추가되어 있음. → 정상. |
| 12 | 펀드 거래 정보와 합계 정보 분리 | 있음 | 분석 정리 문서는 펀드 거래 정보와 합계 정보를 별도 테이블로 분리하여 기재함. → 정상. |
| 13 | 메타데이터 정확성 | 있음 | “KB Bank 설정금액”, “KB Bank 해지금액(LINA 수취금액)”, “KB Bank NET 금액”은 원문과 일치. → 정상. |
| 14 | 투자일임운용사 상세 | 있음 | 원문에 “- Korea Inv,  Shinhan, Samsung”으로 공백 2개 존재. 분석 정리 문서는 “Korea Inv,  Shinhan, Samsung”으로 공백 1개로 정규화. → 원문은 공백 2개이므로, “Korea Inv,  Shinhan, Samsung”으로 수정해야 함. (공백 2개 유지) |
| 15 | 문서 제목 | 있음 | “변액일임펀드 설정/해지 지시서”가 최상단에 기재됨. → 정상. |
| 16 | Markdown 오류 | 없음 | 테이블 구조가 올바르게 작성됨. | 
| 17 | 기타: 합계 정보 테이블의 운용사명 누락 | 있음 | 합계 정보 테이블의 마지막 행 “합계(운용사)”의 운용사명은 “합계(운용사)”로 기재되어 있으나, 원문에서는 “합계(운용사)”가 운용사명 컬럼에 위치함. → 정상. |
| 18 | 펀드잔여좌수 계산 오류 | 있음 | “삼성자산”의 “유럽주식형” 펀드: 전일좌수(3489683178) + 설정신청좌수(17594) - 해지좌수(59667661) = 3430033111 → 정확. → 정상. |
| 19 | 설정/해지금액 합계 오류 | 있음 | “삼성자산” 합계의 설정금액: 1,243,750 + 19,500 + 1,106,000 + 146,250 = 2,515,500 → 원문과 일치. → 정상. |
| 20 | 미처리금액 합계 오류 (재확인) | 있음 | 원문의 “합계(운용사)” 행의 미처리금액은 1,316,057,991이며, 이는 “유럽주식형” 펀드의 미처리금액과 동일. → 합계 행에 이 값을 반영해야 함. 분석 정리 문서는 0으로 잘못 기재. → **심각한 오류**. |

2. 검수 결과 점수  
VALIDATE_RESULT:FAIL

### 검수 결과 오류 재 수정

In [23]:
fixed_markdown = text_to_markdown_error_fix_with_llm(fixed_markdown.content, validate_report.content, _original_text)
# print(fixed_markdown)

########## 오류 수정 prompt ##########

    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서에 대한 분석 정리 문서를 검수하여, 검수 결과를 정리한 검수 결과 보고서입니다.

    검수 결과 보고서에 작성된 오류 사항을 확인하여 오류가 있으면 변액일임펀드 설정/해지 지시서 분석 정리 문서를 수정하세요.    

    오류 수정 시, 정보가 부족하면 변액일임펀드 설정/해지 지시서 분석 정리 문서 작성 지침을 참고하세요.

    오류 수정 시, 정보가 부족하면 변액일임펀드 설정/해지 지시서 원본 파일 문서를 참고하세요.

    아래의 출력 형식에 따라 출력하세요.

    # 검수 결과 보고서

    [1. 검수 결과 보고서

| 일련번호 | 검수 항목 | 검수 내용 | 근거 및 수정 사항 |
|----------|-----------|-----------|------------------|
| 1 | 한자 변환 | 없음 | 원문에 한자가 전혀 발견되지 않음. |
| 2 | 메타데이터 누락 | 있음 | 원문에 “Prepared by”, “Checked by”, “Approved by”가 존재하나 분석 정리 문서에 누락됨. 이 항목들은 문서의 공식적 검토 프로세스를 나타내므로 반드시 포함되어야 함. |
| 3 | 펀드명 공백/띄어쓰기 오류 | 있음 | “브이캡그로스형  ” (2행, 18번)에 끝에 공백 1개 존재. 원문에도 동일한 공백이 있으나, 의미상 오류로 판단되어 제거해야 함. → “브이캡그로스형”으로 수정. |
| 4 | 운용사명 병합 오류 | 있음 | 원문에서 “삼성자산,한투”는 두 운용사의 공동 운용을 의미하나, 분석 정리 문서에서는 “삼성자산,한투”로 정상 기재됨. 그러나 “삼성자산_한투_합계”라는 합계 행이 원문에 존재함에도 분석 정리 문서에서는 누락됨. → 합계 행을 추가해야 함. |
| 5 | 합계 행 누락 | 있음 | 원문에 “삼성자산_한투_합계” 합계 행이 존재함 (행 번호: 삼성자산,한투 다음). 그러

In [24]:
display(Markdown(fixed_markdown.content))

[[1. 메타 데이터 정리 테이블

| 일련번호 | 항목 | 값 | 비고 |
|----------|------|----|------|
| 1 | 문서 제목 | 변액일임펀드 설정/해지 지시서 | 시트명: 당일 |
| 2 | Closing Date | 2025-08-25 00:00:00 | 결제 전 기준일 |
| 3 | 결제일 | 2025-08-26 00:00:00 | 실제 자금 결제일 |
| 4 | 발신자 | 라이나생명보험주식회사 | |
| 5 | 집합투자업자(Collective Investment Business Entity) | LINA Insurance Co. | |
| 6 | 신탁업자(Trust Business Entity) | KB Bank | |
| 7 | 일반사무관리회사(General Administration Company) | HANA Investors Service | |
| 8 | 투자일임운용사(Discretionary Investment Asset Mgt. Co.) | 3 AMC | |
| 9 | 투자일임운용사 상세 | Korea Inv,  Shinhan, Samsung | |
| 10 | Prepared by |  | 원문에 존재하나 값 누락 |
| 11 | Checked by |  | 원문에 존재하나 값 누락 |
| 12 | Approved by |  | 원문에 존재하나 값 누락 |
| 13 | KB Bank 설정금액 | 1959157539 | 수익자(라이나)가 지급할 금액 |
| 14 | KB Bank 해지금액(LINA 수취금액) | 446673328 | 수익자(라이나)가 수취할 금액 |
| 15 | KB Bank NET 금액 | 1512484211 | 설정금액 - 해지금액 (KB Bank 기준 정산액) |

2. 펀드 거래 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 6104 | 채권형 | KB은행 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 | 0 | 0 |
| 2 | 삼성자산 | 6108 | 유럽주식형 | KB은행 | 3489683178 | 17594 | 59667661 | 1108.34 | 19500 | 66132056 | 3430033111 | 20250826 | 1187413601 | 1316057991 |
| 3 | 삼성자산 | 6114 | 인덱스주식형 | KB은행 | 1792484830 | 427550 | 20361 | 2586.83 | 1106000 | 52671 | 1792892019 | 20250826 | 0 | 0 |
| 4 | 삼성자산 | 6115 | 삼성그룹주형 | KB은행 | 436582138 | 101709 | 3119 | 1437.93 | 146250 | 4485 | 436680728 | 20250826 | 0 | 0 |
| 5 | 삼성자산,한투 | 6101 | 혼합성장형 | KB은행 | 8141715654 | 5687732 | 1240755 | 4556.44 | 25915800 | 5653417 | 8146162631 | 20250826 | 0 | 0 |
| 6 | 신한BNP | 6105 | 해외혼합형 | KB은행 | 282648356 | 154566 | 2883 | 1712.86 | 264750 | 4939 | 282800039 | 20250826 | 0 | 0 |
| 7 | 신한BNP | 6106 | 애그리비즈니스주식형 | KB은행 | 1373108730 | 1104931 | 15977 | 1054.41 | 1165050 | 16846 | 1374197684 | 20250826 | 0 | 0 |
| 8 | 신한BNP | 6107 | 아시아50주식형 | KB은행 | 1406890167 | 465143 | 375119 | 2106.88 | 980000 | 790329 | 1406980191 | 20250826 | 0 | 0 |
| 9 | 신한BNP | 6109 | 기후변화주식형 | KB은행 | 52079311 | 0 | 0 | 1737.47 | 0 | 0 | 52079311 | 20250826 | 0 | 0 |
| 10 | 신한BNP | 610J | 밸류고배당주식형 | KB은행 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 | 0 | 0 |
| 11 | 신한BNP | 610L | 미국주식형 | KB은행 | 2032544521 | 507226 | 432988 | 1757.6 | 891500 | 761019 | 2032618759 | 20250826 | 0 | 0 |
| 12 | 신한BNP | 6110 | 차이나주식형 | KB은행 | 15125229377 | 2662577910 | 426437881 | 721.37 | 1920703825 | 307619493 | 17361369406 | 20250826 | 0 | 0 |
| 13 | 신한BNP | 6111 | 글로벌이머징형 | KB은행 | 1471152392 | 805855 | 13021 | 1529.71 | 1232725 | 19919 | 1471945226 | 20250826 | 0 | 0 |
| 14 | 신한BNP | 6112 | 이머징커머더티주식형 | KB은행 | 548143851 | 0 | 841 | 1508.2 | 0 | 1268 | 548143010 | 20250826 | 0 | 0 |
| 15 | 신한BNP | 6113 | 주식형 | KB은행 | 2386104162 | 573999 | 8471011 | 2895.04 | 1661750 | 24523917 | 2378207150 | 20250826 | 0 | 0 |
| 16 | 신한BNP | 6118 | 동남아시아형 | KB은행 | 16270639 | 0 | 0 | 1113.8 | 0 | 0 | 16270639 | 20250826 | 0 | 0 |
| 17 | 한국투신 | 6102 | 혼합안정형 | KB은행 | 1643221367 | 1446095 | 126965 | 2646.19 | 3826639 | 335969 | 1644540497 | 20250826 | 0 | 0 |
| 18 | 한국투신 | 6103 | 브이캡그로스형 | KB은행 | 174005603 | 0 | 6340 | 1520.38 | 0 | 9639 | 173999263 | 20250826 | 0 | 0 |
| 19 | 한국투신 | 610K | 해외혼합&시니어론형 | KB은행 | 31248561 | 0 | 0 | 1185.38 | 0 | 0 | 31248561 | 20250826 | 0 | 0 |
| 20 | 한국투신 | 6116 | 스마트&세이프코스피원자재형 | KB은행 | 357834644 | 0 | 16211 | 1116.98 | 0 | 18107 | 357818433 | 20250826 | 0 | 0 |
| 21 | 한국투신 | 6117 | 스마트&세이프코스피항셍형 | KB은행 | 119098188 | 0 | 0 | 1052.83 | 0 | 0 | 119098188 | 20250826 | 0 | 0 |

3. 합계 정보 정리 테이블

| 일련번호 | 운용사명 | 펀드코드 | 펀드명 | 수탁사명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 | 미처리좌수 | 미처리금액 |
|----------|----------|----------|--------|----------|----------|--------------|----------|----------|----------|----------|--------------|----------|------------|------------|
| 1 | 삼성자산 | 삼성자산_합계 |  |  | 11098676039 | 1265800 | 82791031 | 0 | 2515500 | 106151097 | 11017150808 |  | 1187413601 | 1316057991 |
| 2 | 삼성자산,한투 | 삼성자산_한투_합계 |  |  | 8141715654 | 5687732 | 1240755 | 0 | 25915800 | 5653417 | 8146162631 |  | 0 | 0 |
| 3 | 신한BNP | 신한BNP_합계 |  |  | 28150335341 | 2666189630 | 436212580 | 0 | 1926899600 | 334505099 | 30380312391 |  | 0 | 0 |
| 4 | 한국투신 | 한국투신_합계 |  |  | 2325408363 | 1446095 | 149516 | 0 | 3826639 | 363715 | 2326704942 |  | 0 | 0 |
| 5 | 합계(운용사) |  |  |  | 49716135397 | 2674589257 | 520393882 | 0 | 1959157539 | 446673328 | 51870330772 |  | 1187413601 | 1316057991 | ]]

### 확정분/청구분 분류

In [25]:
classify_confirmed_expected_data = classify_confirmed_expected_from_instruction_with_llm(fixed_markdown.content)
# print(fixed_markdown)

########## 확정분/청구분 구분 prompt ##########

   아래의 제공된 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시서 내용을 분석하여 markdown 형식으로 작성한 문서입니다.

   제공된 문서의 내용을 아래의 확정분과 청구분 분류 기준에 따라 확정분과 청구분 그리고 거래없음으로 분류하여 정리하세요. 

   정리된 결과를 아래의 출력 내용에 따라 출력하세요.   


   # 확정분과 청구분 분류 기준

   [
  목표: 문서 텍스트만으로 확정분/청구분/검토필요/거래없음을 "필드(컬럼 값)" 단위로 안정적으로 분리한다.

  핵심 원칙
  - (R1) 분류 단위는 "필드(컬럼 값)"이다. (행/펀드 단위 아님)
  - (R2) 근거는 반드시 "해당 필드가 속한 컬럼그룹/섹션" 범위로만 서술한다.
  - (R3) 라벨링(확정/청구/검토/거래없음)과 출력(정규화/적재)을 분리한다.
  - (R4) 문서에 없는 값을 생성/치환/역할변경하지 않는다. (공백→0 금지, 작성일→결제일 치환 금지, 검수일 등 임의 생성 금지)
  - (R4-2) raw_value는 원문 그대로 보존한다. 정정은 canonical_value로만 수행한다.
  - (R5) NAV/Price 부재만으로 청구분을 단정하지 않는다.
  - (R6) 확정/청구 라벨과 품질태그(QualityTag)는 독립이다.
  - (R7) 거래유형(설정/해지 등)은 방향, 확정/청구는 시간/상태다.
  - (R8) 컬럼명 표준화는 display_label로만 하고 raw_label은 내부 추적용으로만 보존한다.

  ------------------------------------------------------------
  -1) RowType 선분리(필수, 최우선)
  - FUND_ROW: 실제 펀드/상품 단위 행(펀드코드/상품코드 식별자 유효)
  - SUMMARY_ROW: 합계/소계/Total/XXX_합계/합계(운용사) 등 집계행
  - META_ROW: 문서 메타데이

In [26]:
display(Markdown(classify_confirmed_expected_data.content))

1. 매입통보일 / 환매신청일  
2025-08-25

2. 전체 거래 집계 현황 테이블  
| 구분 | 설정금액 | 해지금액 | 설정좌수 | 해지좌수 |
|------|----------|----------|----------|----------|
| 전체 | 1959157539 | 446673328 | 2674589257 | 520393882 |

3. 전체 펀드 코드 개수 현황 테이블  
3-1. 거래없음 펀드 코드 개수  
2  
3-2. 확정분 펀드 코드 개수  
17  
3-3. 청구분 펀드 코드 개수  
1  

4. 전체 거래 개수 집계 현황 테이블  
4-1. 거래없음 거래 개수  
0  
4-2. 확정분 거래 개수  
34  
4-3. 청구분 거래 개수  
2  

5. 거래없음 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 설정신청좌수 | 해지좌수 | 설정금액 | 해지금액 | 미처리좌수 | 미처리금액 |
|----------|----------|--------|--------------|----------|----------|----------|------------|------------|
| 신한BNP | 6109 | 기후변화주식형 | 0 | 0 | 0 | 0 | 0 | 0 |
| 한국투신 | 6116 | 스마트&세이프코스피원자재형 | 0 | 16211 | 0 | 18107 | 0 | 0 |

6. 확정분 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 |
|----------|----------|--------|----------|--------------|----------|----------|----------|----------|--------------|----------|
| 삼성자산 | 6104 | 채권형 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 |
| 삼성자산 | 6108 | 유럽주식형 | 3489683178 | 17594 | 59667661 | 1108.34 | 19500 | 66132056 | 3430033111 | 20250826 |
| 삼성자산 | 6114 | 인덱스주식형 | 1792484830 | 427550 | 20361 | 2586.83 | 1106000 | 52671 | 1792892019 | 20250826 |
| 삼성자산 | 6115 | 삼성그룹주형 | 436582138 | 101709 | 3119 | 1437.93 | 146250 | 4485 | 436680728 | 20250826 |
| 삼성자산,한투 | 6101 | 혼합성장형 | 8141715654 | 5687732 | 1240755 | 4556.44 | 25915800 | 5653417 | 8146162631 | 20250826 |
| 신한BNP | 6105 | 해외혼합형 | 282648356 | 154566 | 2883 | 1712.86 | 264750 | 4939 | 282800039 | 20250826 |
| 신한BNP | 6106 | 애그리비즈니스주식형 | 1373108730 | 1104931 | 15977 | 1054.41 | 1165050 | 16846 | 1374197684 | 20250826 |
| 신한BNP | 6107 | 아시아50주식형 | 1406890167 | 465143 | 375119 | 2106.88 | 980000 | 790329 | 1406980191 | 20250826 |
| 신한BNP | 6110 | 밸류고배당주식형 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 |
| 신한BNP | 6111 | 글로벌이머징형 | 1471152392 | 805855 | 13021 | 1529.71 | 1232725 | 19919 | 1471945226 | 20250826 |
| 신한BNP | 6112 | 이머징커머더티주식형 | 548143851 | 0 | 841 | 1508.2 | 0 | 1268 | 548143010 | 20250826 |
| 신한BNP | 6113 | 주식형 | 2386104162 | 573999 | 8471011 | 2895.04 | 1661750 | 24523917 | 2378207150 | 20250826 |
| 신한BNP | 6118 | 동남아시아형 | 16270639 | 0 | 0 | 1113.8 | 0 | 0 | 16270639 | 20250826 |
| 한국투신 | 6102 | 혼합안정형 | 1643221367 | 1446095 | 126965 | 2646.19 | 3826639 | 335969 | 1644540497 | 20250826 |
| 한국투신 | 6103 | 브이캡그로스형 | 174005603 | 0 | 6340 | 1520.38 | 0 | 9639 | 173999263 | 20250826 |
| 한국투신 | 610K | 해외혼합&시니어론형 | 31248561 | 0 | 0 | 1185.38 | 0 | 0 | 31248561 | 20250826 |
| 한국투신 | 6117 | 스마트&세이프코스피항셍형 | 119098188 | 0 | 0 | 1052.83 | 0 | 0 | 119098188 | 20250826 |

7. 청구분 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 미처리좌수 | 미처리금액 |
|----------|----------|--------|------------|------------|
| 삼성자산 | 6108 | 유럽주식형 | 1187413601 | 1316057991 |

### 확정분/청구분 검수

In [27]:
classify_confirmed_expected_validate_report = classify_confirmed_expected_validate_report_with_llm(
    classify_confirmed_expected_data.content, 
    fixed_markdown.content
)
# print(fixed_markdown)

########## 확정분/청구분 검수 prompt ##########

   아래의 제공된 '확정분과 청구분 분류 결과 문서'는 제공된 '변액일임펀드 설정/해지 지시서'를 분석하여 확정분과 청구분으로 분류한 문서입니다.

   제공된 '확정분과 청구분 분류 결과 문서'가 아래의 '확정분과 청구분 분류 기준'에 따라 정확하게 작성되었는지 여부를 아래의 검수 지침에 따라 검수하세요. 

   검수가 완료되면 아래의 출력 형식에 따라 검수 결과 보고서를 작성하여 출력하세요.



   # 확정분과 청구분 분류 기준

   [
  목표: 문서 텍스트만으로 확정분/청구분/검토필요/거래없음을 "필드(컬럼 값)" 단위로 안정적으로 분리한다.

  핵심 원칙
  - (R1) 분류 단위는 "필드(컬럼 값)"이다. (행/펀드 단위 아님)
  - (R2) 근거는 반드시 "해당 필드가 속한 컬럼그룹/섹션" 범위로만 서술한다.
  - (R3) 라벨링(확정/청구/검토/거래없음)과 출력(정규화/적재)을 분리한다.
  - (R4) 문서에 없는 값을 생성/치환/역할변경하지 않는다. (공백→0 금지, 작성일→결제일 치환 금지, 검수일 등 임의 생성 금지)
  - (R4-2) raw_value는 원문 그대로 보존한다. 정정은 canonical_value로만 수행한다.
  - (R5) NAV/Price 부재만으로 청구분을 단정하지 않는다.
  - (R6) 확정/청구 라벨과 품질태그(QualityTag)는 독립이다.
  - (R7) 거래유형(설정/해지 등)은 방향, 확정/청구는 시간/상태다.
  - (R8) 컬럼명 표준화는 display_label로만 하고 raw_label은 내부 추적용으로만 보존한다.

  ------------------------------------------------------------
  -1) RowType 선분리(필수, 최우선)
  - FUND_ROW: 실제 펀드/상품 단위 행(펀드코드/상품코드 식별자 유효)
  - SUMMARY_ROW: 합계/소계/Total

In [28]:
display(Markdown(classify_confirmed_expected_validate_report.content))

1. 검수 결과 보고서

| 검수 항목 | 검수 내용 | 결과 | 근거 |
|-----------|-----------|------|------|
| 거래없음 펀드 분류 정확성 | 거래없음으로 분류된 펀드(신한BNP 6109, 한국투신 6116)의 설정/해지 좌수 및 금액이 모두 0인지 확인 | ✅ 정확 | 신한BNP 6109: 설정좌수=0, 해지좌수=0, 설정금액=0, 해지금액=0<br>한국투신 6116: 설정좌수=0, 해지금액=0, 해지좌수=16211 → **오류!**<br>→ 해지좌수=16211, 해지금액=18107로 **거래 존재**하므로 거래없음 분류 불가 |
| 거래없음 펀드 개수 일치성 | 거래없음 펀드 코드 개수: 분류 결과 2개 vs 실제 거래없음 펀드 수 | ❌ 불일치 | 분류 결과: 2개 (6109, 6116)<br>실제: 6109는 거래없음(정상), 6116은 해지좌수/금액 존재 → **거래없음은 1개만 정상**<br>→ 분류 결과 2개는 오류, 실제 거래없음 펀드는 **1개** |
| 청구분 분류 정확성 | 청구분 데이터(삼성자산 6108)의 미처리좌수/금액이 존재하고, 설정/해지 금액/좌수도 존재하는지 확인 | ✅ 정확 | 삼성자산 6108: 설정신청좌수=17594, 해지좌수=59667661, 설정금액=19500, 해지금액=66132056 (모두 >0) + 미처리좌수=1,187,413,601, 미처리금액=1,316,057,991 → **청구분(A) 정당** |
| 확정분 분류 정확성 | 확정분 펀드(17개)의 설정/해지 금액/좌수가 모두 >0 또는 0이지만 미처리 없고, 기준가격/결제일/잔고 일관성 확인 | ✅ 정확 | 모든 확정분 펀드는 미처리좌수/금액=0, 설정/해지 금액/좌수 중 일부는 0이지만 (예: 신한BNP 6110, 6112, 6118, 한국투신 6103, 610K, 6117) → **설정/해지 중 하나만 0이어도 확정분 가능**<br>→ 기준가격, 결제일, 전일/잔여좌수 롤포워드 일관성 확인 가능 (예: 전일좌수 + 설정 - 해지 = 잔여좌수) → **E/E2 규칙 만족** |
| 확정분 펀드 개수 일치성 | 확정분 펀드 코드 개수: 분류 결과 17개 vs 실제 확정분 펀드 수 | ✅ 일치 | 지시서에서 총 21개 펀드 행 중, 거래없음 1개(6109), 청구분 1개(6108) → 21 - 1 - 1 = 19개<br>→ 그러나 분류 결과는 17개 → **불일치!**<br>→ **분류 결과 누락: 신한BNP 610J(밸류고배당), 610L(미국주식형), 6110(차이나주식형)**<br>→ 이 3개 펀드는 설정/해지 금액/좌수 존재, 미처리=0 → **확정분이어야 함**<br>→ 분류 결과에 누락됨 → **17개 → 20개가 정확** |
| 청구분 거래 개수 일치성 | 청구분 거래 개수: 분류 결과 2건 vs 실제 청구분 필드 수 | ❌ 불일치 | 분류 결과: 청구분 거래 개수=2<br>실제: 청구분은 **1개 펀드**(6108)에 **미처리좌수, 미처리금액** 2개 필드 존재 → **2개 필드** → **정확**<br>→ 그러나 분류 결과 테이블은 1행에 2개 필드 → **거래 건수는 1개 펀드, 2개 필드** → **"거래 개수" 정의가 모호**<br>→ 기준: "청구분 거래 개수" = **청구분 필드 수** (R1: 분류 단위는 필드) → **2개 필드 → 2건으로 정확** |
| 확정분 거래 개수 일치성 | 확정분 거래 개수: 분류 결과 34건 vs 실제 확정분 필드 수 | ❌ 불일치 | 분류 결과: 34건<br>실제 확정분 펀드: 17개 펀드 × 6개 필드(설정신청좌수, 해지좌수, 설정금액, 해지금액, 전일좌수, 펀드잔여좌수, 결제일) → **7개 필드**<br>→ 17 × 7 = 119개 필드 → **34건은 너무 적음**<br>→ **기준: 확정분 거래 개수 = TXN_EXEC + BAL + DATE + PRICE 필드 수**<br>→ TXN_EXEC: 설정신청좌수, 해지좌수, 설정금액, 해지금액 → 4개<br>→ BAL: 전일좌수, 펀드잔여좌수 → 2개<br>→ DATE, PRICE → 2개<br>→ 총 8개 필드 × 17 펀드 = **136개 필드**<br>→ 분류 결과 34건은 **정확하지 않음**<br>→ **분류 결과의 "거래 개수"는 필드 수가 아니라 펀드 행 수로 오해한 듯**<br>→ 기준(R1): 분류 단위는 **필드** → **34건은 오류** |
| 합계 정합성 검증 | 전체 설정금액/해지금액/좌수 합계가 지시서와 일치하는지 | ✅ 일치 | 지시서 합계: 설정금액=1,959,157,539 / 해지금액=446,673,328 / 설정좌수=2,674,589,257 / 해지좌수=520,393,882<br>→ 분류 결과 테이블과 동일 → **정합성 만족** |
| 펀드코드 누락 | 확정분 펀드에 누락된 펀드코드 존재 여부 | ❌ 존재 | 지시서에 있는 확정분 펀드 중, 분류 결과에 누락된 펀드:<br>- 신한BNP 610J (밸류고배당주식형)<br>- 신한BNP 610L (미국주식형)<br>- 신한BNP 6110 (차이나주식형)<br>→ 이 3개는 설정/해지 금액/좌수 존재, 미처리=0 → **확정분이어야 함**<br>→ 분류 결과에 **누락** → **중대 오류** |
| 거래없음 펀드 오분류 | 한국투신 6116이 거래없음으로 분류된 오류 | ❌ 존재 | 한국투신 6116: 해지좌수=16,211, 해지금액=18,107 → **거래 존재**<br>→ **청구분 또는 확정분으로 분류되어야 함**<br>→ 미처리=0 → **확정분**<br>→ **거래없음으로 오분류** → **중대 오류** |
| 출력 형식 오류 | Markdown 테이블 형식 오류 여부 | ✅ 정상 | 모든 테이블이 올바른 Markdown 형식으로 작성됨. 열 구분자, 행 구분자 정상. |

2. 검수 결과 점수  
VALIDATE_RESULT:FAIL

### 확정분/청구분 오류 수정

In [29]:
classify_confirmed_expected_fix_data = classify_confirmed_expected_error_fix_with_llm(
    classify_confirmed_expected_validate_report.content,
    classify_confirmed_expected_data.content, 
    fixed_markdown.content
)
# print(fixed_markdown)

########## 확정분/청구분 오류 수정 prompt ##########

    아래는 '확정분과 청구분 분류 기준'에 따라 확정분과 청구분으로 분류한 '확정분과 청구분 분류 결과 문서'를 아래의 '검수 지침'에 따라 검수하여, 검수 결과를 정리한 '검수 결과 보고서'입니다.

    검수 결과 보고서에 작성된 오류 사항을 확인하여 오류가 있으면 '확정분과 청구분 분류 결과 문서'를 수정하세요.    

    오류 수정 시, 정보가 부족하면 '확정분과 청구분 분류 기준'을 참고하세요.

    오류 수정 시, 정보가 부족하면 '변액일임펀드 설정/해지 지시서'를 참고하세요.

    오류 수정 시, 정보가 부족하면 '검수 지침'을 참고하세요.

    아래의 출력 형식에 따라 출력하세요.



    # 검수 결과 보고서

    [1. 검수 결과 보고서

| 검수 항목 | 검수 내용 | 결과 | 근거 |
|-----------|-----------|------|------|
| 거래없음 펀드 분류 정확성 | 거래없음으로 분류된 펀드(신한BNP 6109, 한국투신 6116)의 설정/해지 좌수 및 금액이 모두 0인지 확인 | ✅ 정확 | 신한BNP 6109: 설정좌수=0, 해지좌수=0, 설정금액=0, 해지금액=0<br>한국투신 6116: 설정좌수=0, 해지금액=0, 해지좌수=16211 → **오류!**<br>→ 해지좌수=16211, 해지금액=18107로 **거래 존재**하므로 거래없음 분류 불가 |
| 거래없음 펀드 개수 일치성 | 거래없음 펀드 코드 개수: 분류 결과 2개 vs 실제 거래없음 펀드 수 | ❌ 불일치 | 분류 결과: 2개 (6109, 6116)<br>실제: 6109는 거래없음(정상), 6116은 해지좌수/금액 존재 → **거래없음은 1개만 정상**<br>→ 분류 결과 2개는 오류, 실제 거래없음 펀드는 **1개** |
| 청구분 분류 정확성 | 청구분 데이터(삼성자산 6108)의 미처리좌수/금액이 존재하고, 설정/해지 금액/좌수도 존재하는지

In [30]:
display(Markdown(classify_confirmed_expected_fix_data.content))

[1. 매입통보일 / 환매신청일  
2025-08-25

2. 전체 거래 집계 현황 테이블  
| 구분 | 설정금액 | 해지금액 | 설정좌수 | 해지좌수 |
|------|----------|----------|----------|----------|
| 전체 | 1959157539 | 446673328 | 2674589257 | 520393882 |

3. 전체 펀드 코드 개수 현황 테이블  
3-1. 거래없음 펀드 코드 개수  
1  
3-2. 확정분 펀드 코드 개수  
20  
3-3. 청구분 펀드 코드 개수  
1  

4. 전체 거래 개수 집계 현황 테이블  
4-1. 거래없음 거래 개수  
0  
4-2. 확정분 거래 개수  
160  
4-3. 청구분 거래 개수  
2  

5. 거래없음 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 설정신청좌수 | 해지좌수 | 설정금액 | 해지금액 | 미처리좌수 | 미처리금액 |
|----------|----------|--------|--------------|----------|----------|----------|------------|------------|
| 신한BNP | 6109 | 기후변화주식형 | 0 | 0 | 0 | 0 | 0 | 0 |

6. 확정분 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 전일좌수 | 설정신청좌수 | 해지좌수 | 기준가격 | 설정금액 | 해지금액 | 펀드잔여좌수 | 결제일 |
|----------|----------|--------|----------|--------------|----------|----------|----------|----------|--------------|----------|
| 삼성자산 | 6104 | 채권형 | 5379925893 | 718947 | 23099890 | 1729.96 | 1243750 | 39961885 | 5357544950 | 20250826 |
| 삼성자산 | 6114 | 인덱스주식형 | 1792484830 | 427550 | 20361 | 2586.83 | 1106000 | 52671 | 1792892019 | 20250826 |
| 삼성자산 | 6115 | 삼성그룹주형 | 436582138 | 101709 | 3119 | 1437.93 | 146250 | 4485 | 436680728 | 20250826 |
| 삼성자산,한투 | 6101 | 혼합성장형 | 8141715654 | 5687732 | 1240755 | 4556.44 | 25915800 | 5653417 | 8146162631 | 20250826 |
| 신한BNP | 6105 | 해외혼합형 | 282648356 | 154566 | 2883 | 1712.86 | 264750 | 4939 | 282800039 | 20250826 |
| 신한BNP | 6106 | 애그리비즈니스주식형 | 1373108730 | 1104931 | 15977 | 1054.41 | 1165050 | 16846 | 1374197684 | 20250826 |
| 신한BNP | 6107 | 아시아50주식형 | 1406890167 | 465143 | 375119 | 2106.88 | 980000 | 790329 | 1406980191 | 20250826 |
| 신한BNP | 6110 | 밸류고배당주식형 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 |
| 신한BNP | 6111 | 글로벌이머징형 | 1471152392 | 805855 | 13021 | 1529.71 | 1232725 | 19919 | 1471945226 | 20250826 |
| 신한BNP | 6112 | 이머징커머더티주식형 | 548143851 | 0 | 841 | 1508.2 | 0 | 1268 | 548143010 | 20250826 |
| 신한BNP | 6113 | 주식형 | 2386104162 | 573999 | 8471011 | 2895.04 | 1661750 | 24523917 | 2378207150 | 20250826 |
| 신한BNP | 6118 | 동남아시아형 | 16270639 | 0 | 0 | 1113.8 | 0 | 0 | 16270639 | 20250826 |
| 한국투신 | 6102 | 혼합안정형 | 1643221367 | 1446095 | 126965 | 2646.19 | 3826639 | 335969 | 1644540497 | 20250826 |
| 한국투신 | 6103 | 브이캡그로스형 | 174005603 | 0 | 6340 | 1520.38 | 0 | 9639 | 173999263 | 20250826 |
| 한국투신 | 610K | 해외혼합&시니어론형 | 31248561 | 0 | 0 | 1185.38 | 0 | 0 | 31248561 | 20250826 |
| 한국투신 | 6117 | 스마트&세이프코스피항셍형 | 119098188 | 0 | 0 | 1052.83 | 0 | 0 | 119098188 | 20250826 |
| 한국투신 | 6116 | 스마트&세이프코스피원자재형 | 357834644 | 0 | 16211 | 1116.98 | 0 | 18107 | 357818433 | 20250826 |
| 신한BNP | 610J | 밸류고배당주식형 | 3456163835 | 0 | 462859 | 1657.89 | 0 | 767369 | 3455700976 | 20250826 |
| 신한BNP | 610L | 미국주식형 | 2032544521 | 507226 | 432988 | 1757.6 | 891500 | 761019 | 2032618759 | 20250826 |
| 신한BNP | 6110 | 차이나주식형 | 15125229377 | 2662577910 | 426437881 | 721.37 | 1920703825 | 307619493 | 17361369406 | 20250826 |

7. 청구분 데이터 테이블  
| 운용사명 | 펀드코드 | 펀드명 | 미처리좌수 | 미처리금액 |
|----------|----------|--------|------------|------------|
| 삼성자산 | 6108 | 유럽주식형 | 1187413601 | 1316057991 |